# Experiment: Timing Script

CIFAR-10 timing experiment: six permutations {training, training+testing} x {baseline, neurodegenration, neurogeneration}

Timed for `TIMING_REPEATS = 25` recorded epochs after `WARMUP_EPOCHS = 5` unrecorded epoch.

The final summary table is saved to `./DataTables/Exp_Timing_Summary.tex`.

In [1]:
from Dependencies import *
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import pickle as pkl
import os
import copy
import time
import platform
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [2]:
TIMING_REPEATS = 25
WARMUP_EPOCHS = 5
NEUROADAPTATION_LAYER = 1

LEARNING_RATE = 1e-3
BATCH_SIZE = 48
DEVICE = try_gpu(output=True, i=0)
ARCHITECTURE = [3072, 500, 500, 10]
PSI_DECAY = 1e-1
ADAMW_WEIGHT_DECAY = 1e-3
NORMALISATION = True
INTRINSIC_LENGTH_APPROACH = "TRAINABLE"
LINEAR_CORRECTION_APPROACH = "TRAINABLE+DECAY"
WEIGHT_INIT = "orthogonal"



DATA_TABLE_DIR = Path("./FinalResults/Exp0Results")
DATA_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using device: {DEVICE}")
print(f"Architecture: {ARCHITECTURE}")
print(f"Timing repeats per case: {TIMING_REPEATS}")
print(f"Warm-up epochs per case: {WARMUP_EPOCHS}")
print(f"Neuroadaptation layer: {NEUROADAPTATION_LAYER}")

GPU Name: NVIDIA GeForce RTX 3080
Memory Usage:
	Total:		 9.6 GB
	Allocated:	 0.0 GB
	Cached:		 0.0 GB

Using device: cuda:0
Architecture: [3072, 500, 500, 10]
Timing repeats per case: 25
Warm-up epochs per case: 5
Neuroadaptation layer: 1


In [3]:
# Found that elementwise normalisation of the dataset was generally beneficial as a preprocessing step.
class PerPixelNormalize:
    def __init__(self):
        if not os.path.exists("./CIFAR_normalisations.pkl"):
            print("Creating ./CIFAR_normalisations.pkl")
            raw_transform = transforms.Compose([transforms.ToTensor()])
            cifar_for_stats = datasets.CIFAR10(root="./data", train=True, download=True, transform=raw_transform)
            cifar_stack = torch.stack([image for image, label in cifar_for_stats], dim=0)
            cifar_mean = cifar_stack.mean(dim=0).to(torch.float32)
            cifar_std = cifar_stack.std(dim=0, unbiased=False).to(torch.float32)
            cifar_inv_std = torch.where(cifar_std > 1e-8, 1.0 / cifar_std, torch.ones_like(cifar_std))
            pkl.dump({"mean": cifar_mean, "inverse stddev": cifar_inv_std}, open("./CIFAR_normalisations.pkl", "wb"))

        normaliser_dictionary = pkl.load(open("./CIFAR_normalisations.pkl", "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor):
        return (tensor - self.mean) * self.inv_std


if NORMALISATION:
    print("Using CIFAR-10 per-pixel normalisation")
    transform = transforms.Compose([transforms.ToTensor(), PerPixelNormalize()])
else:
    print("Not using normalisation")
    transform = transforms.Compose([transforms.ToTensor()])

cifar_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
cifar_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(cifar_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(cifar_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train examples: {len(cifar_train)}")
print(f"Test examples:  {len(cifar_test)}")
print(f"Train batches:  {len(train_loader)}")
print(f"Test batches:   {len(test_loader)}")

Using CIFAR-10 per-pixel normalisation
Train examples: 50000
Test examples:  10000
Train batches:  1042
Test batches:   209


In [4]:
def synchronise_if_needed(device=DEVICE):
    """Synchronise CUDA kernels so wall-clock timings include queued GPU work."""
    if isinstance(device, torch.device) and device.type == "cuda":
        torch.cuda.synchronize(device)


def time_call(fn, *, device=DEVICE):
    synchronise_if_needed(device)
    start = time.perf_counter()
    output = fn()
    synchronise_if_needed(device)
    elapsed = time.perf_counter() - start
    return output, elapsed


def get_device_facts(device=DEVICE):
    facts = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device": str(device),
    }
    if isinstance(device, torch.device) and device.type == "cuda":
        idx = device.index if device.index is not None else torch.cuda.current_device()
        props = torch.cuda.get_device_properties(idx)
        facts.update({
            "gpu_index": idx,
            "gpu_name": torch.cuda.get_device_name(idx),
            "total_memory_gb": props.total_memory / 1024 ** 3,
            "multi_processor_count": props.multi_processor_count,
            "cuda_runtime_version": torch.version.cuda,
            "cudnn_version": torch.backends.cudnn.version(),
        })
    return facts


def print_device_facts(device=DEVICE):
    facts = get_device_facts(device)
    print("Device facts")
    for key, value in facts.items():
        if isinstance(value, float):
            print(f"  {key}: {value:.3f}")
        else:
            print(f"  {key}: {value}")
    return facts


def print_timing_method():
    method = (
        "Timing method: each measured block uses time.perf_counter(). "
        "If CUDA is active, torch.cuda.synchronize() is called immediately before and after "
        "each measured block, so queued GPU kernels are included in wall-clock time. "
        f"{WARMUP_EPOCHS} warm-up epochs per case are excluded from the summary."
    )
    print(method)
    return method


# THis remains standard code from my original dependencies though.
def build_network_and_optimiser(initial_state_dict=None):
    network = IsotropicTanhMLP(
        layers=ARCHITECTURE,
        flatten=True,
        unflatten_shape=None,
        intrinsic_length_approach=INTRINSIC_LENGTH_APPROACH,
        linear_correction_approach=LINEAR_CORRECTION_APPROACH,
        positive_intrinsic_length=True,
        init_intrinsic_length=1e-6,
        tanh_epsilon=1e-3,
        device=DEVICE,
        dtype=torch.get_default_dtype(),
    )
    if initial_state_dict is None:
        network.simple_initialiser(weight_init=WEIGHT_INIT)
    else:
        network.load_state_dict(copy.deepcopy(initial_state_dict))
    network.to(DEVICE)

    optimiser = torch.optim.AdamW(
        network.parameters(),
        lr=LEARNING_RATE,
        weight_decay=ADAMW_WEIGHT_DECAY,
    )
    return network, optimiser


# ChatGPT needless helper function.
def apply_procedure(network, optimiser, procedure):
    if procedure == "baseline":
        return
    if procedure == "neurodegeneration":
        network.neurodegenerate(layer=NEUROADAPTATION_LAYER, optimiser=optimiser)
        return
    if procedure == "neurogeneration":
        network.neurogenerate(layer=NEUROADAPTATION_LAYER, optimiser=optimiser)
        return
    raise ValueError(f"Unknown procedure: {procedure}")

In [5]:
device_facts = print_device_facts(DEVICE)
timing_method = print_timing_method()

loss = nn.CrossEntropyLoss()
lambda_psi = PSI_DECAY if LINEAR_CORRECTION_APPROACH.upper() == "TRAINABLE+DECAY" else 0.0

# Using one initial initialisation for comparability
base_network, _ = build_network_and_optimiser()
BASE_INITIAL_STATE_DICT = copy.deepcopy(base_network.state_dict())
del base_network

TIMING_CASES = [
    {"case": "baseline_train", "with_testing": False, "procedure": "baseline"},
    {"case": "baseline_train_test", "with_testing": True, "procedure": "baseline"},
    {"case": "neurodegeneration_train", "with_testing": False, "procedure": "neurodegeneration"},
    {"case": "neurodegeneration_train_test", "with_testing": True, "procedure": "neurodegeneration"},
    {"case": "neurogeneration_train", "with_testing": False, "procedure": "neurogeneration"},
    {"case": "neurogeneration_train_test", "with_testing": True, "procedure": "neurogeneration"},
]

raw_timing_records = []

# Build one independently evolving network/optimiser per case, then interleave cases by epoch.
case_states = {}
for case in TIMING_CASES:
    case_states[case["case"]] = build_network_and_optimiser(BASE_INITIAL_STATE_DICT)

total_epochs = WARMUP_EPOCHS + TIMING_REPEATS

# Standard training loop:
for epoch in range(total_epochs):
    is_warmup = epoch < WARMUP_EPOCHS
    repeat = epoch - WARMUP_EPOCHS + 1

    for case in TIMING_CASES:
        network, optimiser = case_states[case["case"]]
        print(f"\n========== {case['case']} ==========")

        cycle_start = time.perf_counter()

        (network, train_x, train_cost, train_acc), training_elapsed = time_call(
            lambda: training_epoch(
                network=network,
                training_set=train_loader,
                device=DEVICE,
                optimiser=optimiser,
                loss=loss,
                current_epoch=epoch,
                classification_or_reconstruction="classification",
                lambda_psi=lambda_psi,
            ),
            device=DEVICE,
        )

        testing_elapsed = 0.0
        test_cost = np.nan
        test_acc = np.nan
        if case["with_testing"]:
            (network, test_cost, test_acc), testing_elapsed = time_call(
                lambda: testing_epoch(
                    network=network,
                    testing_set=test_loader,
                    device=DEVICE,
                    loss=loss,
                    classification_or_reconstruction="classification",
                ),
                device=DEVICE,
            )

        procedure_elapsed = 0.0
        architecture_before_procedure = list(network.architecture)
        if case["procedure"] != "baseline":
            _, procedure_elapsed = time_call(
                lambda: apply_procedure(network, optimiser, case["procedure"]),
                device=DEVICE,
            )
        architecture_after_procedure = list(network.architecture)
        case_states[case["case"]] = (network, optimiser)

        synchronise_if_needed(DEVICE)
        total_elapsed = time.perf_counter() - cycle_start

        if not is_warmup:
            record = {
                "case": case["case"],
                "procedure": case["procedure"],
                "with_testing": case["with_testing"],
                "repeat": repeat,
                "training_elapsed_seconds": training_elapsed,
                "testing_elapsed_seconds": testing_elapsed,
                "procedure_elapsed_seconds": procedure_elapsed,
                "total_elapsed_seconds": total_elapsed,
                "train_cost_mean": float(np.mean(train_cost)),
                "train_acc_mean": float(np.mean(train_acc)),
                "test_cost": float(test_cost) if case["with_testing"] else np.nan,
                "test_acc": float(test_acc) if case["with_testing"] else np.nan,
                "architecture_before_procedure": architecture_before_procedure,
                "architecture_after_procedure": architecture_after_procedure,
            }
            raw_timing_records.append(record)
            print(
                f"repeat {repeat:02d}/{TIMING_REPEATS} | "
                f"train={training_elapsed:.3f}s | "
                f"test={testing_elapsed:.3f}s | "
                f"procedure={procedure_elapsed:.3f}s | "
                f"total={total_elapsed:.3f}s | "
                f"arch={architecture_after_procedure}"
            )
        else:
            print(f"warm-up complete | arch={architecture_after_procedure}")

raw_timing_df = pd.DataFrame(raw_timing_records)
raw_timing_df

Device facts
  python: 3.11.14
  platform: Linux-7.2.6-arch2-1-x86_64-with-glibc2.44
  torch: 2.6.0+cu124
  cuda_available: True
  device: cuda:0
  gpu_index: 0
  gpu_name: NVIDIA GeForce RTX 3080
  total_memory_gb: 9.640
  multi_processor_count: 68
  cuda_runtime_version: 12.4
  cudnn_version: 90100
Timing method: each measured block uses time.perf_counter(). If CUDA is active, torch.cuda.synchronize() is called immediately before and after each measured block, so queued GPU kernels are included in wall-clock time. 5 warm-up epochs per case are excluded from the summary.

========== baseline_train ==========
Done!
warm-up complete | arch=[3072, 500, 500, 10]

========== baseline_train_test ==========
Done!
warm-up complete | arch=[3072, 500, 500, 10]

========== neurodegeneration_train ==========
Done!
warm-up complete | arch=[3072, 500, 499, 10]

========== neurodegeneration_train_test ==========
Done!
warm-up complete | arch=[3072, 500, 499, 10]

========== neurogeneration_train ===

,case,procedure,with_testing,repeat,training_elapsed_seconds,testing_elapsed_seconds,procedure_elapsed_seconds,total_elapsed_seconds,train_cost_mean,train_acc_mean,test_cost,test_acc,architecture_before_procedure,architecture_after_procedure
0,baseline_train,baseline,False,1,17.726251,0.000000,0.000000,17.726466,1.632237,43.042228,NaN,NaN,"[3072, 500, 500, 10]","[3072, 500, 500, 10]"
1,baseline_train_test,baseline,True,1,17.374475,1.751838,0.000000,19.126497,1.633651,43.023234,1.668786,41.447370,"[3072, 500, 500, 10]","[3072, 500, 500, 10]"
2,neurodegeneration_train,neurodegeneration,False,1,17.902256,0.000000,0.041694,17.944100,1.638598,42.998242,NaN,NaN,"[3072, 500, 495, 10]","[3072, 500, 494, 10]"
3,neurodegeneration_train_test,neurodegeneration,True,1,18.041649,1.738302,0.041988,19.822125,1.641021,42.802304,1.680397,40.968901,"[3072, 500, 495, 10]","[3072, 500, 494, 10]"
4,neurogeneration_train,neurogeneration,False,1,17.331100,0.000000,0.029494,17.360809,1.638028,43.143195,NaN,NaN,"[3072, 500, 505, 10]","[3072, 500, 506, 10]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,baseline_train_test,baseline,True,25,16.966260,1.694633,0.000000,18.661051,1.449451,48.968332,1.594365,44.039076,"[3072, 500, 500, 10]","[3072, 500, 500, 10]"
146,neurodegeneration_train,neurodegeneration,False,25,17.493163,0.000000,0.032928,17.526267,1.464360,48.457495,NaN,NaN,"[3072, 500, 471, 10]","[3072, 500, 470, 10]"
147,neurodegeneration_train_test,neurodegeneration,True,25,18.006521,1.712268,0.032050,19.751034,1.466668,48.310542,1.592515,43.560607,"[3072, 500, 471, 10]","[3072, 500, 470, 10]"
148,neurogeneration_train,neurogeneration,False,25,17.555918,0.000000,0.030639,17.586749,1.469807,48.354528,NaN,NaN,"[3072, 500, 529, 10]","[3072, 500, 530, 10]"


In [6]:
metric_columns = [
    "training_elapsed_seconds",
    "testing_elapsed_seconds",
    "procedure_elapsed_seconds",
    "total_elapsed_seconds",
]

summary_numeric = (
    raw_timing_df
    .groupby(["case", "procedure", "with_testing"], as_index=False)[metric_columns]
    .agg(["mean", "sem", "min", "max"])
)
summary_numeric.columns = ["_".join(col).strip("_") for col in summary_numeric.columns.to_flat_index()]
summary_numeric = summary_numeric.reset_index()

summary_for_tex = summary_numeric[[
    "case",
    "procedure",
    "with_testing",
    "training_elapsed_seconds_mean",
    "training_elapsed_seconds_sem",
    "testing_elapsed_seconds_mean",
    "testing_elapsed_seconds_sem",
    "procedure_elapsed_seconds_mean",
    "procedure_elapsed_seconds_sem",
    "total_elapsed_seconds_mean",
    "total_elapsed_seconds_sem",
]].copy()

for base in [
    "training_elapsed_seconds",
    "testing_elapsed_seconds",
    "procedure_elapsed_seconds",
    "total_elapsed_seconds",
]:
    summary_for_tex[base.replace("_elapsed_seconds", "_seconds")] = (
        summary_for_tex[f"{base}_mean"].map(lambda x: f"{x:.3f}")
        + " $\\pm$ "
        + summary_for_tex[f"{base}_sem"].fillna(0.0).map(lambda x: f"{x:.3f}")
    )

latex_table = summary_for_tex[[
    "case",
    "procedure",
    "with_testing",
    "training_seconds",
    "testing_seconds",
    "procedure_seconds",
    "total_seconds",
]].to_latex(
    index=False,
    escape=False,
    caption=f"CIFAR-10 timing summary. Values are mean $\\pm$ standard error over {TIMING_REPEATS} recorded epochs per case; {WARMUP_EPOCHS} warm-up epochs per case are excluded.",
    label="tab:experiment_timing_summary",
)

tex_path = DATA_TABLE_DIR / "Exp_Timing_Summary.tex"
csv_path = DATA_TABLE_DIR / "Exp_Timing_Raw.csv"
summary_csv_path = DATA_TABLE_DIR / "Exp_Timing_Summary.csv"

tex_path.write_text(latex_table)
raw_timing_df.to_csv(csv_path, index=False)
summary_numeric.to_csv(summary_csv_path, index=False)

print(f"Saved LaTeX summary table to: {tex_path}")
print(f"Saved raw timing CSV to: {csv_path}")
print(f"Saved numeric summary CSV to: {summary_csv_path}")

summary_for_tex[[
    "case",
    "procedure",
    "with_testing",
    "training_seconds",
    "testing_seconds",
    "procedure_seconds",
    "total_seconds",
]]

Saved LaTeX summary table to: FinalResults/Exp0Results/Exp_Timing_Summary.tex
Saved raw timing CSV to: FinalResults/Exp0Results/Exp_Timing_Raw.csv
Saved numeric summary CSV to: FinalResults/Exp0Results/Exp_Timing_Summary.csv


,case,procedure,with_testing,training_seconds,testing_seconds,procedure_seconds,total_seconds
0,baseline_train,baseline,False,17.523 $\pm$ 0.062,0.000 $\pm$ 0.000,0.000 $\pm$ 0.000,17.524 $\pm$ 0.062
1,baseline_train_test,baseline,True,17.495 $\pm$ 0.097,1.738 $\pm$ 0.003,0.000 $\pm$ 0.000,19.234 $\pm$ 0.098
2,neurodegeneration_train,neurodegeneration,False,17.778 $\pm$ 0.074,0.000 $\pm$ 0.000,0.037 $\pm$ 0.001,17.815 $\pm$ 0.074
3,neurodegeneration_train_test,neurodegeneration,True,17.891 $\pm$ 0.088,1.749 $\pm$ 0.006,0.036 $\pm$ 0.001,19.676 $\pm$ 0.091
4,neurogeneration_train,neurogeneration,False,17.560 $\pm$ 0.063,0.000 $\pm$ 0.000,0.030 $\pm$ 0.000,17.591 $\pm$ 0.063
5,neurogeneration_train_test,neurogeneration,True,17.555 $\pm$ 0.071,1.741 $\pm$ 0.004,0.030 $\pm$ 0.000,19.327 $\pm$ 0.073
